In [ ]:
!pip install opencv-python
import cv2
import matplotlib.pyplot as plt
import os
import numpy as np
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

#### Fonction pour déterminer le numéro de frame

In [ ]:
def num_frame(path_video):
    # Ouvrir la vidéo
    video_capture = cv2.VideoCapture(path_video)

    # Liste pour stocker les coordonnées des cercles
    coord = []

    # Liste pour stocker les numéros de frame avec des cercles détectés
    frames_with_circles = []

    # Initialisation du numéro de frame
    frame_number = 0

    while True:
        # Lire une image de la vidéo
        ret, frame = video_capture.read()

        # Vérifier si la lecture de la vidéo est terminée
        if not ret:
            print("Lecture de la vidéo terminée.")
            break

        # Convertir l'image en niveaux de gris
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Appliquer un lissage pour réduire le bruit
        blurred = cv2.GaussianBlur(gray, (9, 9), 2)

        # Détecter les cercles avec la méthode de HoughCircles
        circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1, minDist=50, param1=50, param2=30, minRadius=10, maxRadius=30)

        # Si des cercles sont détectés, les dessiner sur l'image et enregistrer le numéro de frame
        if circles is not None:
            circles = np.uint16(np.around(circles))
            for i in circles[0, :]:
                # Dessiner le cercle détecté
                cv2.circle(frame, (i[0], i[1]), i[2], (0, 255, 0), 2)
                # Dessiner le centre du cercle
                cv2.circle(frame, (i[0], i[1]), 2, (0, 0, 255), 3)
                center_coordinates = (i[0], i[1])
                coord.append(center_coordinates)

                # Enregistrer le numéro de frame
                frames_with_circles.append(frame_number)

        # Afficher la vidéo avec les cercles détectés
        #cv2.imshow('Cercles détectés', frame)

        # Attendre l'appui sur la touche 'q' pour quitter
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        # Incrémenter le numéro de frame
        frame_number += 1

    # Libérer la capture vidéo et fermer les fenêtres
    video_capture.release()
    cv2.destroyAllWindows()

    # Retourner le numéro de la première frame avec des cercles détectés
    return frames_with_circles[0]


#### Fonction de détermination de xc et yc

In [ ]:
def xc_yc(path_video, num_frame):
    chemin_video = path_video

    # Ouvrir la vidéo en utilisant OpenCV
    cap = cv2.VideoCapture(chemin_video)

    coord1 = []

    # Vérifier si la vidéo est correctement ouverte
    if not cap.isOpened():
        print("Erreur: Impossible d'ouvrir la vidéo.")
        return coord1

    # Aller à la frame spécifiée
    cap.set(cv2.CAP_PROP_POS_FRAMES, num_frame)

    # Lire la frame
    ret, frame = cap.read()

    # Vérifier si la lecture de la frame a réussi
    if not ret:
        print("Erreur: Impossible de lire la frame.")
        cap.release()
        return coord1

    # Convertir l'image en niveaux de gris
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Appliquer un lissage pour réduire le bruit
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)

    # Détecter les cercles avec la méthode de HoughCircles
    circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1, minDist=50, param1=50, param2=30, minRadius=10, maxRadius=30)

    # Si des cercles sont détectés, les dessiner sur l'image
    if circles is not None:
        circles = np.uint16(np.around(circles))
        for i in circles[0, :]:
            # Dessiner le cercle détecté
            cv2.circle(frame, (i[0], i[1]), i[2], (0, 255, 0), 2)
            # Dessiner le centre du cercle
            cv2.circle(frame, (i[0], i[1]), 2, (0, 0, 255), 3)
            center_coordinates = (i[0], i[1])
            coord1.append(center_coordinates)
    else:
        center_coordinates = (-1, -1)  # Si aucun cercle n'est détecté
        coord1.append(center_coordinates)

    cv2.waitKey(0)

    # Fermer la fenêtre
    cv2.destroyAllWindows()

    # Libérer la capture vidéo
    cap.release()

    return coord1


In [ ]:
# Définir une fonction gaussienne
def gaussian(x, a, x0, sigma):
    return a * np.exp(-(x - x0)**2 / (2 * sigma**2))

#### Fonction de détermination du seuil

In [ ]:
def calcul_seuil(path_video, num_frame, x, y):
    chemin_video = path_video
    largeur = 600
    hauteur = 100

    # Ouvrir la vidéo en utilisant OpenCV
    cap = cv2.VideoCapture(chemin_video)

    # Vérifier si la vidéo est correctement ouverte
    if not cap.isOpened():
        print("Erreur: Impossible d'ouvrir la vidéo.")
        return None

    # Aller à la frame spécifiée
    cap.set(cv2.CAP_PROP_POS_FRAMES, num_frame)

    # Lire la frame
    ret, frame = cap.read()

    # Vérifier si la lecture de la frame a réussi
    if not ret:
        print("Erreur: Impossible de lire la frame.")
        cap.release()
        return None

    # Fermer la vidéo
    cap.release()

    # Convertir l'image en niveaux de gris
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray_reg = gray[y:y + hauteur, x:x + largeur]

    # Calculer l'histogramme
    histogramme = cv2.calcHist([gray_reg], [0], None, [256], [0, 256])
    data = np.squeeze(histogramme)

    # Trouver les pics dans l'histogramme
    peaks, _ = find_peaks(data, height=0)

    # Filtrer les pics avant l'abscisse 50
    peaks_before_50 = [peak for peak in peaks if peak < 50]

    # Trouver le plus grand pic parmi ceux avant l'abscisse 50
    if peaks_before_50:
        max_peak_before_50 = max(peaks_before_50, key=lambda x: data[x])
    else:
        print("Aucun pic détecté avant l'abscisse 50.")
        return None

    p = max_peak_before_50

    # Récupérer les données autour du pic détecté
    window_size = 10  # Ajustez la taille de la fenêtre autour du pic
    x_window = np.arange(max(0, p - window_size), min(len(data), p + window_size))
    y_window = data[x_window]

    # Ajuster une gaussienne aux données
    initial_guess = [np.max(y_window), np.mean(x_window), 1]  # Estimations initiales des paramètres
    popt, _ = curve_fit(gaussian, x_window, y_window, p0=initial_guess)

    # Extraire l'écart type de la gaussienne ajustée
    sigma_gaussian = popt[2]
    seuil = p + round(sigma_gaussian) + 1

    return seuil

In [ ]:
# Chemin du dossier contenant les vidéos
dossier_videos = 'pfa2/vid'

# Chemin du dossier où sauvegarder les résultats
dossier_resultat = "C:/Users/nadam/OneDrive/Bureau/Résultat_poursuite"

# Vérifier si le dossier résultat existe, sinon le créer
if not os.path.exists(dossier_resultat):
    os.makedirs(dossier_resultat)

# Liste tous les fichiers vidéo dans le dossier
videos = [f for f in os.listdir(dossier_videos) if f.endswith('.avi')]

# Parcourir chaque vidéo dans le dossier
for video_filename in videos:
    # Construire le chemin complet du fichier vidéo
    chemin_video = os.path.join(dossier_videos, video_filename)
     # Créer un sous-dossier pour sauvegarder les résultats de chaque vidéo
    nom_sous_dossier = os.path.splitext(video_filename)[0]
    dossier_resultat_video = os.path.join(dossier_resultat, nom_sous_dossier)
    os.makedirs(dossier_resultat_video, exist_ok=True)

    # Ouvrir la vidéo
    cap = cv2.VideoCapture(chemin_video)

    # Vérifier si la vidéo est correctement ouverte
    if not cap.isOpened():
        print("Erreur: Impossible d'ouvrir la vidéo:", chemin_video)
        continue
         # Configurations pour l'enregistrement de la vidéo résultante
    fourcc = cv2.VideoWriter_fourcc(*'X264')
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Chemin de la vidéo résultante
    chemin_video_resultat = os.path.join(dossier_resultat_video, "video_resultante.mp4")
    out = cv2.VideoWriter(chemin_video_resultat, fourcc, fps, (width, height))


    # Récupérer le numéro de la frame à partir de la fonction num_frame
    nf = num_frame(chemin_video)

    # Récupérer les coordonnées (xc, yc) à partir de la fonction xc_yc
    l = xc_yc(chemin_video, nf)
    xc = l[0][0]
    yc = l[0][1]

    # Calculer le seuil à partir de la fonction calcul_seuil
    largeur = 600
    hauteur = 100
    x = int(xc - (largeur / 4))
    y = int(yc - (hauteur / 2))
    seuil = calcul_seuil(chemin_video, nf, x, y)

    # Créer un sous-dossier pour sauvegarder les résultats de chaque vidéo
    nom_sous_dossier = os.path.splitext(video_filename)[0]
    dossier_resultat_video = os.path.join(dossier_resultat, nom_sous_dossier)
    os.makedirs(dossier_resultat_video, exist_ok=True)

    # Réinitialiser la capture vidéo
    cap = cv2.VideoCapture(chemin_video)

    coord = []
    while True:
        # Lire une frame de la vidéo
        ret, frame = cap.read()

        # Vérifier si la fin de la vidéo est atteinte
        if not ret:
            print("Fin de la vidéo.")
            break

        # Convertir l'image en niveaux de gris
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Créer une copie de la frame pour afficher la vidéo originale avec les contours et le centre détecté
        frame_contours = frame.copy()

        # Appliquer le seuillage binaire inverse uniquement à la région spécifiée
        region_to_threshold_inv = gray_frame[y:y + hauteur, x:x + largeur]
        _, thresholded_region_inv = cv2.threshold(region_to_threshold_inv, seuil, 255, cv2.THRESH_BINARY_INV)
        thresholded_region = cv2.medianBlur(thresholded_region_inv, 13)

        # Créer une copie de la frame originale pour afficher la vidéo seuillée uniquement dans la région définie
        frame_thresholded = frame.copy()
        frame_thresholded[y:y + hauteur, x:x + largeur] = cv2.merge([thresholded_region, thresholded_region, thresholded_region])

        # Trouver les contours à l'intérieur de la région seuillée inversement
        contours, _ = cv2.findContours(thresholded_region, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = sorted(contours, key=cv2.contourArea, reverse=True)
        # Tracer un rectangle englobant autour de chaque contour
        for contour in contours:
            xx, yy, w, h = cv2.boundingRect(contour)
            cv2.rectangle(frame_contours, (xx + x, yy + y), (xx + x + w, yy + y + h), (0, 255, 0), 2)
            # Calculer les coordonnées du centre
            center_x = xx + w // 2 + x
            center_y = yy + h // 2 + y
            # Dessiner un cercle (point) au centre
            cv2.circle(frame_contours, (center_x, center_y), 3, (255, 0, 0), -1)  # Bleu (BGR)
            coord.append([center_x, center_y])

        # Affichage des vidéos
        cv2.imshow('Vidéo originale avec contours et centre détecté', frame_contours)
        cv2.imshow('Vidéo seuillée uniquement dans la région définie', frame_thresholded)

        # Arrêt lorsque la touche 'q' est pressée
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Libérer la capture vidéo
    cap.release()
    cv2.destroyAllWindows()

    # Enregistrement des résultats dans le sous-dossier correspondant
    # (sauvegarde de la courbe de poursuite et de la liste d'abscisse)
    x_coords = [c[0] for c in coord]
    abscisse = [(x - x_coords[0]) for x in x_coords]
    for i in range(len(abscisse)):
        if abscisse[i]>80:
            abscisse[i]=abscisse[i-1]
    temps = np.arange(len(abscisse)) / 25
    plt.figure(figsize=(10, 6))
    plt.plot(temps, abscisse)
    plt.title('Suivi de la pupille')
    plt.xlabel('Temps (s)')
    plt.ylabel('Position (pixels)')
    plt.savefig(os.path.join(dossier_resultat_video, "Courbe_de_poursuite.png"))
    plt.close()

    np.savetxt(os.path.join(dossier_resultat_video, "abscisse_liste.txt"), abscisse)

print("Terminé.")

Lecture de la vidéo terminée.
Fin de la vidéo.
Lecture de la vidéo terminée.
Fin de la vidéo.
Lecture de la vidéo terminée.
Terminé.
